In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg  , udf ,StringType
# Initialize
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()
# Create Base DataFrame
data = [
    (1, "Alice", "Engineering", 75000, 25),
    (2, "Bob", "Marketing", 60000, 30),
    (3, "Charlie", "Engineering", 80000, 35),
    (4, "David", "Sales", 65000, 28),
    (5, "Eve", "Marketing", 70000, 32)
]
columns = ["Id", "Name", "Department", "Salary", "Age"]
df = spark.createDataFrame(data, columns);

In [8]:
df_select = df.select("Name","Department", "Salary")

In [9]:
df_select.show()

+-------+-----------+------+
|   Name| Department|Salary|
+-------+-----------+------+
|  Alice|Engineering| 75000|
|    Bob|  Marketing| 60000|
|Charlie|Engineering| 80000|
|  David|      Sales| 65000|
|    Eve|  Marketing| 70000|
+-------+-----------+------+



In [10]:
df_filter= df.filter(col("Salary")>65000)

In [11]:
df_filter.show()

+---+-------+-----------+------+---+
| Id|   Name| Department|Salary|Age|
+---+-------+-----------+------+---+
|  1|  Alice|Engineering| 75000| 25|
|  3|Charlie|Engineering| 80000| 35|
|  5|    Eve|  Marketing| 70000| 32|
+---+-------+-----------+------+---+



In [13]:
df_col = df.withColumn("Bonus",col("Salary")*0.10)

In [14]:
df_col.show()

+---+-------+-----------+------+---+------+
| Id|   Name| Department|Salary|Age| Bonus|
+---+-------+-----------+------+---+------+
|  1|  Alice|Engineering| 75000| 25|7500.0|
|  2|    Bob|  Marketing| 60000| 30|6000.0|
|  3|Charlie|Engineering| 80000| 35|8000.0|
|  4|  David|      Sales| 65000| 28|6500.0|
|  5|    Eve|  Marketing| 70000| 32|7000.0|
+---+-------+-----------+------+---+------+



In [15]:
df.write.mode("overwrite").csv("Output/employees",header=True)

In [4]:
def can_drive(age):
    if age>=18:
        return "Can Drive"
    elif age>=13 :
        return "Drive with Learning Licence"
    else:
        return "Cannot Drive"

In [5]:
data=[("A",25),("B",14),("C",11),("D",20),("E",8)]
columns=["Name","Age"]
d1=spark.createDataFrame(data,columns)

In [6]:
d1.show()

+----+---+
|Name|Age|
+----+---+
|   A| 25|
|   B| 14|
|   C| 11|
|   D| 20|
|   E|  8|
+----+---+



In [7]:
drive_udf=udf(can_drive,StringType())

In [8]:
df_method=d1.withColumn("drive",drive_udf(col("Age")))
print("method-1 ")
df_method.show()

method-1 


+----+---+--------------------+
|Name|Age|               drive|
+----+---+--------------------+
|   A| 25|           Can Drive|
|   B| 14|Drive with Learni...|
|   C| 11|        Cannot Drive|
|   D| 20|           Can Drive|
|   E|  8|        Cannot Drive|
+----+---+--------------------+



In [9]:
spark.udf.register("sql_drive",can_drive,StringType())
d1.createOrReplaceTempView("people")

In [10]:
sql_df=spark.sql('''
select Name, Age , sql_drive(Age) as DriveAPPlicable from people ''')
sql_df.show()

+----+---+--------------------+
|Name|Age|     DriveAPPlicable|
+----+---+--------------------+
|   A| 25|           Can Drive|
|   B| 14|Drive with Learni...|
|   C| 11|        Cannot Drive|
|   D| 20|           Can Drive|
|   E|  8|        Cannot Drive|
+----+---+--------------------+

